In [4]:
library("lme4")
library("margins")
library("stargazer")
library("emmeans")
library("ggeffects")
library("broom")
library("broom.mixed")
library("MASS")
library("pscl")
library("fixest")
library("marginaleffects")
library("modelsummary")
library("glmmTMB")
library("dplyr")

In [3]:
main_path <- "/home/20250114zmz_kd/"
data <- read.csv(paste0(main_path, "UseBSForNot/MergeInsthData_1211.csv"))
dim(data)

[1] 3227892      78

In [5]:
print(names(data))

 [1] "X"                                           
 [2] "work_id"                                     
 [3] "fac_pub"                                     
 [4] "year"                                        
 [5] "paper_type"                                  
 [6] "paper_language"                              
 [7] "nwbin"                                       
 [8] "new_word_reuse_times"                        
 [9] "npbin"                                       
[10] "new_phrase_reuse_times"                      
[11] "novelbin"                                    
[12] "rao_stirling"                                
[13] "author_id"                                   
[14] "author_position"                             
[15] "institution_id"                              
[16] "country_code"                                
[17] "country"                                     
[18] "num_author_log"                              
[19] "num_inst_log"                                
[20] "num_co

In [6]:
colSums(is.na(data))

X 
                                           0 
                                     work_id 
                                           0 
                                     fac_pub 
                                           0 
                                        year 
                                           0 
                                  paper_type 
                                           0 
                              paper_language 
                                           0 
                                       nwbin 
                                           0 
                        new_word_reuse_times 
                                           0 
                                       npbin 
                                           0 
                      new_phrase_reuse_times 
                                           0 
                                    novelbin 
                                           0 
                                rao_stirling 
                                      303430 
                                   author_id 
                                           0 
                             author_position 
                                           0 
                              institution_id 
                                           0 
                                country_code 
                                           0 
                                     country 
                                           0 
                              num_author_log 
                                           0 
                                num_inst_log 
                                           0 
                             num_country_log 
                                           0 
                           num_reference_log 
                                           0 
                             timescited5_log 
                                      107147 
                            timescited10_log 
                                      163886 
                           timescitedall_log 
                                           0 
                               ab_length_log 
                                           0 
                     leadership_global_class 
                                           0 
                         mean_career_age_log 
                                           0 
                            inst_h_index_log 
                                           6 
                                   source_id 
                                           0 
                          source_h_index_log 
                                           0 
                                 source_type 
                                           0 
                                 core_source 
                                           0 
        Agricultural.and.Biological.Sciences 
                                           0 
                         Arts.and.Humanities 
                                           0 
Biochemistry..Genetics.and.Molecular.Biology 
                                           0 
         Business..Management.and.Accounting 
                                           0 
                        Chemical.Engineering 
                                           0 
                                   Chemistry 
                                           0 
                            Computer.Science 
                                           0 
                           Decision.Sciences 
                                           0 
                                   Dentistry 
                                           0 
                Earth.and.Planetary.Sciences 
                                           0 
         Economics..Econometrics.and.Finance 
                                           0 
                                      Energy 
                                         

In [7]:
# 把所有无限值替换成 NA
data[sapply(data, is.infinite)] <- NA

In [8]:
data <- data[!is.na(data$Atyp_10pct_Z), ]
dim(data)
data <- data[data$num_reference >=5, ]
dim(data)

[1] 2814103      78

[1] 2760385      78

In [9]:
# data <- data[data$ab_length > 0, ]
# dim(data)
data <- data[data$year >= 1950, ]
dim(data)

[1] 2760385      78

In [10]:
# 找出所有包含无限值的行和列
inf_mask <- sapply(data, function(col) is.infinite(col))
rows_with_inf <- apply(inf_mask, 1, any)  # 哪些行至少有一个Inf
cols_with_inf <- colnames(data)[apply(inf_mask, 2, any)]  # 哪些列有Inf

# 打印包含无限值的行数和列名
cat("包含无限值的行数:", sum(rows_with_inf), "\n")
cat("包含无限值的列名:", paste(cols_with_inf, collapse = ", "), "\n")

# 查看这些行具体内容
data_filt_with_inf <- data[rows_with_inf, c(cols_with_inf), drop=FALSE]
print(data_filt_with_inf)

包含无限值的行数: 0 
包含无限值的列名:  
data frame with 0 columns and 0 rows


In [11]:
data$decade <- (data$year %/% 10) * 10

In [12]:
data$leadership_global_class <- factor(data$leadership_global_class)
data <- within(data, leadership_global_class <- relevel(leadership_global_class, ref = 'shared'))
data$fac_pub <- factor(data$fac_pub)
data <- within(data, fac_pub <- relevel(fac_pub, ref = 'NonBSF'))
data$core_source <- factor(data$core_source)
data <- within(data, core_source <- relevel(core_source, ref = 'NonCore'))
data$decade <- as.factor(data$decade)

In [13]:
# variable type
paper_level <- "num_author_log + num_inst_log + num_country_log + num_reference_log + leadership_global_class + mean_career_age_log + inst_h_index_log" 
journal_level <- "source_h_index_log + core_source"
disciplines <- "Arts.and.Humanities + Biochemistry..Genetics.and.Molecular.Biology + Business..Management.and.Accounting + Chemical.Engineering + 
 Chemistry + Computer.Science + Decision.Sciences + Dentistry + Earth.and.Planetary.Sciences + 
Economics..Econometrics.and.Finance + Energy + Engineering + Environmental.Science + Health.Professions + 
Immunology.and.Microbiology + Materials.Science + Mathematics + Medicine + Neuroscience + Nursing +
Pharmacology..Toxicology.and.Pharmaceutics + Physics.and.Astronomy + Psychology + Social.Sciences + Veterinary "

In [16]:
fml <- as.formula(
  paste0("novelbin ~ fac_pub + ", paper_level, "+", journal_level, "+", disciplines, " + decade | author_id")
)

model_total <- feglm(fml, data = data, family = binomial("logit"), vcov = "iid")
summary(model_total)

NOTES: 4 observations removed because of NA values (RHS: 4).
       17,669 fixed-effects (78,042 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novelbin
Observations: 2,682,339
Fixed-effects: author_id: 61,171
Standard-errors: IID 
                                              Estimate Std. Error    z value
fac_pubBSF                                    0.078407   0.006058  12.942567
num_author_log                                0.251961   0.004571  55.121331
num_inst_log                                  0.006668   0.005830   1.143847
num_country_log                              -0.281281   0.008623 -32.620194
num_reference_log                             0.146210   0.002932  49.863537
leadership_global_classallNorth               0.138223   0.008748  15.800371
leadership_global_classallSouth              -0.013734   0.013781  -0.996557
mean_career_age_log                           0.021688   0.004796   4.521938
inst_h_index_log                             -0.022841   0.003707  -6.161047
source_h_index_log                           -0.041600   0.002180 -19.086730
core_sourceCore    

In [17]:
# 每组 reg_class 的平均预测概率
res_year <- avg_comparisons(model_total, variables = "fac_pub", by = 'year', comparison = "ratio")
res_year

term,contrast,year,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
fac_pub,mean(BSF) / mean(NonBSF),1950,1.043906,0.003489774,299.1327,0,Inf,1.037066,1.050746,0.24952031,0.26448921,0.24952031
fac_pub,mean(BSF) / mean(NonBSF),1951,1.043812,0.003475804,300.3082,0,Inf,1.037000,1.050625,0.51321315,0.53277105,0.51321315
fac_pub,mean(BSF) / mean(NonBSF),1952,1.042042,0.003334492,312.5041,0,Inf,1.035507,1.048578,0.32625672,0.34372069,0.32625672
fac_pub,mean(BSF) / mean(NonBSF),1953,1.041591,0.003297087,315.9124,0,Inf,1.035128,1.048053,0.10596614,0.11362702,0.10596614
fac_pub,mean(BSF) / mean(NonBSF),1954,1.044018,0.003488701,299.2570,0,Inf,1.037180,1.050856,0.19954549,0.21236472,0.19954549
fac_pub,mean(BSF) / mean(NonBSF),1955,1.042944,0.003407367,306.0851,0,Inf,1.036266,1.049623,0.60607177,0.62462763,0.60607177
fac_pub,mean(BSF) / mean(NonBSF),1956,1.047888,0.003803191,275.5286,0,Inf,1.040434,1.055342,0.29326446,0.30977439,0.29326446
fac_pub,mean(BSF) / mean(NonBSF),1957,1.048483,0.003851646,272.2169,0,Inf,1.040934,1.056032,0.46714847,0.48670608,0.46714847
fac_pub,mean(BSF) / mean(NonBSF),1958,1.044585,0.003533643,295.6114,0,Inf,1.037659,1.051511,0.27392965,0.28979742,0.27392965


In [15]:
MEs = ggemmeans(model_total, terms=c('fac_pub', 'decade'), typical='median')

In [15]:
MEs

x,predicted,std.error,conf.low,conf.high,group
<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>
NonBSF,0.2891848,0.03128525,0.2767451,0.3019501,1950
NonBSF,0.2727920,0.08242563,0.2419434,0.3059862,1960
NonBSF,0.2541376,0.07898041,0.2259253,0.2845777,1970
NonBSF,0.2633962,0.07806457,0.2348015,0.2941346,1980
NonBSF,0.3157658,0.07768517,0.2838265,0.3495452,1990
NonBSF,0.2748942,0.07748536,0.2456776,0.3061751,2000
NonBSF,0.2525434,0.07739032,0.2249978,0.2822332,2010
NonBSF,0.2545190,0.07742528,0.2268113,0.2843667,2020
BSF,0.3055643,0.03130841,0.2927009,0.3187384,1950


In [18]:
fname = paste0(main_path, "UseBSForNot/compare_ns_1211_year.csv")
write.csv(res_year, fname, row.names = FALSE)